# Challenge 5: Deploying an Agent to Agent Platform

**Goal:** Demonstrate the ability to deploy and use an agent using Google Agent Platform.

**Requirements covered in this notebook (builds on Challenge 4):**
1. An agent created using ADK (already built in prior challenges).
2. Deploy the agent to Agent Platform.
3. Test the deployed agent.
4. Uploaded to GitHub for grading.

Everything from Challenges 1–4 is carried over unchanged. New additions are marked with `# === CHALLENGE 5 ENHANCEMENT ===` comments throughout.

**Which agent gets deployed, and why:** `greeter_agent` — it's the most complete pipeline built so far (greeter → `answer_team` SequentialAgent → search → critique → refine), and its entire hierarchy is **Gemini-only** (no `LiteLlm`/third-party model anywhere in that tree), which means the deployed container doesn't need `litellm` or the SAIC gateway credentials at all — meaningfully simpler than deploying the dual-model weather agents would be.

**Deployment mechanism:** this is a notebook, not a CLI session, so deployment uses the Python SDK equivalent of `adk deploy agent_engine` — `vertexai.agent_engines.create()` — rather than the `adk deploy` command used in the GENAI107 lab. Conceptually the same operation (packages the agent, uploads to a GCS staging bucket, builds and deploys a managed container on Agent Engine), just invoked programmatically instead of from the command line.

## Step 0: Setup, Installation, and API Key Management

You'll be prompted for your keys below rather than pasting them into the cell — this keeps real credentials out of the notebook's saved source. `GOOGLE_MAPS_API_KEY` and `PROJECT_ID` come from the Cloud Skills Boost lab environment. `SAIC_API_KEY` comes from your SAIC-provided LLM gateway credentials.

In [ ]:
!pip install google-adk litellm -q
print("Installation complete.")

In [ ]:
import getpass
import os

# Secrets are prompted for (masked input) rather than hardcoded, so they
# never end up sitting in this cell's saved source.
GOOGLE_MAPS_API_KEY = getpass.getpass("Enter your Google Maps API key: ")
SAIC_API_KEY = getpass.getpass("Enter your SAIC API token: ")

# Not a secret, so a plain prompt is fine.
PROJECT_ID = input("Enter your GCP PROJECT_ID: ")

SAIC_API_BASE = "https://ai-api.apps.factory.saic.com"

os.environ["OPENAI_API_KEY"] = SAIC_API_KEY
os.environ["OPENAI_API_BASE"] = SAIC_API_BASE

print("Environment configured.")

## Step 1: Imports, Settings and Constants

In [ ]:
from google.adk.agents import Agent
from google.adk.agents import SequentialAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from google.adk.models import LlmResponse, LlmRequest
from google.adk.agents.callback_context import CallbackContext
from typing import Optional

# Built-in ADK tool for Google Search grounding. Only supported on Gemini
# models (Vertex's native grounding mechanism) — not available via
# LiteLLM/third-party models.
from google.adk.tools import google_search

# AgentTool wraps an agent so it can be used as a regular function tool,
# rather than a sub_agents hierarchy member. Needed for any agent whose
# only tool is google_search — see the Step 9 design note for why plain
# sub_agents doesn't work for a pure-search agent.
from google.adk.tools import AgentTool

# Gemini model (native ADK support, no wrapper needed)
MODEL_GEMINI = "gemini-2.5-flash"

# Third-party model via LiteLLM, routed through the SAIC OpenAI-compatible gateway.
# "bedrock-claude-haiku-4-5" is the cheapest/fastest tier available per SAIC's
# model config — confirm the exact model string matches what the gateway expects.
MODEL_THIRD_PARTY = LiteLlm(model="openai/bedrock-claude-haiku-4-5")

print("Environment configured.")

## Step 2: `get_current_weather(lat, lon)` — National Weather Service Tool

Unchanged from Challenge 1/2/3/4.

In [ ]:
import requests


def get_current_weather(lat: float, lon: float) -> str:
    """Retrieve the current weather forecast for a US location.

    Uses the National Weather Service (NWS) API, which requires a two-stage
    lookup: first resolve the (lat, lon) pair to a forecast-office grid
    square via the /points endpoint, then fetch that grid square's
    time-series forecast and return the most immediate period.

    Args:
        lat: Latitude of the target location. Must fall within the United
            States and its territories.
        lon: Longitude of the target location. Must fall within the United
            States and its territories.

    Returns:
        A human-readable summary combining the current forecast period's
        name and detailed forecast text, e.g. "Tonight: Mostly clear, with
        a low around 55." If the NWS API is unavailable or the coordinates
        are out of range, returns a human-readable error message instead
        of raising, so the calling agent can relay it to the user.
    """
    # NWS API requires a descriptive User-Agent header or it returns 403.
    headers = {"User-Agent": "(agent-dev-skills-workshop, jay.watson@saic.com)"}

    try:
        points_url = f"https://api.weather.gov/points/{lat},{lon}"
        points_response = requests.get(points_url, headers=headers, timeout=10)
        points_response.raise_for_status()
        forecast_url = points_response.json()["properties"]["forecast"]

        forecast_response = requests.get(forecast_url, headers=headers, timeout=10)
        forecast_response.raise_for_status()
        current_period = forecast_response.json()["properties"]["periods"][0]

        return f"{current_period['name']}: {current_period['detailedForecast']}"
    except requests.RequestException as exc:
        return (
            "The National Weather Service is temporarily unavailable "
            f"(error: {exc}). Please try again in a moment."
        )


# Quick manual check (Washington, DC)
# print(get_current_weather(38.8894, -77.0352))

## Step 3: `get_location_lat_long(city, state)` — Google Maps Geocoding Tool

Unchanged from Challenge 1/2/3/4.

In [ ]:
def get_location_lat_long(city: str, state: str) -> dict[str, float | None]:
    """Convert a city and state into latitude/longitude coordinates.

    Uses the Google Maps Geocoding API.

    Args:
        city: The city name, e.g. "Knoxville".
        state: The state name or abbreviation, e.g. "TN" or "Tennessee".

    Returns:
        A dict with "Latitude" and "Longitude" keys. Both values are None
        if the location could not be resolved or the API call failed.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": f"{city}, {state}", "key": GOOGLE_MAPS_API_KEY}

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print(f"Geocoding API error: {response.status_code}")
        return {"Latitude": None, "Longitude": None}

    data = response.json()
    if data.get("status") != "OK" or not data.get("results"):
        print(f"Geocoding API returned no results: {data.get('status')}")
        return {"Latitude": None, "Longitude": None}

    location = data["results"][0]["geometry"]["location"]
    return {"Latitude": location["lat"], "Longitude": location["lng"]}


# Quick manual check
# print(get_location_lat_long("Knoxville", "TN"))

## Step 4: Callback Functions

Unchanged from Challenge 2/3/4.

In [ ]:
def log_user_prompt(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Log the user's most recent message before it's sent to the model.

    Pure observation — always returns None so the model call proceeds
    normally regardless of what's logged.
    """
    last_user_message = ""
    if llm_request.contents and llm_request.contents[-1].role == "user":
        if llm_request.contents[-1].parts:
            last_user_message = llm_request.contents[-1].parts[0].text

    if last_user_message:
        print(f"[{callback_context.agent_name}] user prompt: '{last_user_message}'")
    return None


def log_model_response(
    callback_context: CallbackContext, llm_response: LlmResponse
) -> Optional[LlmResponse]:
    """Log the model's response (text, tool call, or error).

    Pure observation — always returns None so the response is passed
    through unmodified.
    """
    if llm_response.content and llm_response.content.parts:
        part = llm_response.content.parts[0]
        if part.text:
            print(f"[{callback_context.agent_name}] model response: '{part.text[:100]}...'")
        elif part.function_call:
            print(f"[{callback_context.agent_name}] model called tool: '{part.function_call.name}'")
        else:
            print(f"[{callback_context.agent_name}] model response: (no text content)")
    elif llm_response.error_message:
        print(f"[{callback_context.agent_name}] model response error: '{llm_response.error_message}'")
    return None

In [ ]:
# Simple keyword heuristics for a teaching example, not a production-grade
# classifier. Two SEPARATE, independently-documented checks per the
# Challenge 2 assignment's two sub-requirements.

NON_US_LOCATION_KEYWORDS = [
    "england", "london", "united kingdom", "scotland", "france", "paris",
    "germany", "berlin", "japan", "tokyo", "china", "beijing", "canada",
    "toronto", "mexico", "australia", "sydney", "india", "mumbai",
    "russia", "moscow", "mars", "moon",
]

MALICIOUS_INPUT_PATTERNS = [
    "ignore previous instructions",
    "ignore all previous",
    "disregard your instructions",
    "disregard the above",
    "you are now",
    "system prompt",
    "jailbreak",
]


def validate_user_input(
    callback_context: CallbackContext, llm_request: LlmRequest
) -> Optional[LlmResponse]:
    """Validate the user's message before it reaches the model.

    Checks (independently) whether the message mentions a non-US location
    the weather tools can't handle, or looks like a prompt-injection
    attempt. If either check fails, returns an LlmResponse directly —
    which short-circuits the model call entirely for this turn.
    """
    last_user_message = ""
    if llm_request.contents and llm_request.contents[-1].role == "user":
        if llm_request.contents[-1].parts:
            last_user_message = llm_request.contents[-1].parts[0].text

    if not last_user_message:
        return None

    lowered = last_user_message.lower()

    for keyword in NON_US_LOCATION_KEYWORDS:
        if keyword in lowered:
            print(f"[{callback_context.agent_name}] validation failed: non-US location ('{keyword}')")
            return LlmResponse(
                content={
                    "role": "model",
                    "parts": [{
                        "text": (
                            "I can only look up weather for US locations — "
                            "the National Weather Service doesn't cover "
                            "international locations. Try a US city instead!"
                        )
                    }],
                }
            )

    for pattern in MALICIOUS_INPUT_PATTERNS:
        if pattern in lowered:
            print(f"[{callback_context.agent_name}] validation failed: malicious input pattern ('{pattern}')")
            return LlmResponse(
                content={
                    "role": "model",
                    "parts": [{"text": "Sorry, I can't process that request."}],
                }
            )

    return None

## Step 5: Build the Weather Agent (model-agnostic, with callbacks)

Unchanged from Challenge 2/3/4.

In [ ]:
AGENT_INSTRUCTION = """
You are a helpful weather assistant, proud through and through to be from
Tennessee.
When the user asks for the weather in a specific city, use the
'get_location_lat_long' function to get the latitude and longitude of the
city, then pass those coordinates to the 'get_current_weather' tool to get
the weather information.
If a tool returns an error, inform the user politely.
If a tool call succeeds, return a clear weather summary.
Any chance you get, work in a mention of how great the Tennessee Volunteers,
Nashville SC, or the Tennessee Titans are - keep it brief and natural, not
forced into every single response, but let your Tennessee pride show.
"""


def build_weather_agent(name: str, model, disallow_transfer_to_peers: bool = False) -> Agent:
    """Build a weather agent instance for the given model.

    Args:
        name: Unique agent name (must differ across instances in the same
            session, and across the whole notebook — every Agent object
            needs its own instance even if two share a name-prefix).
        model: Either a Gemini model string, or a LiteLlm-wrapped model for
            third-party providers.
        disallow_transfer_to_peers: Set True when this agent will sit
            alongside sibling sub-agents under a shared parent (e.g. next to
            search_agent). Without this, a sticky transfer_to_agent handoff
            can leave this agent active for a later off-topic question,
            which would otherwise attempt a sideways peer transfer straight
            to a sibling — crashing with the search/non-search tool-mixing
            error, since this agent's own tools would get pulled into that
            sibling's isolated call. Forcing transfers back up through the
            (tool-less) parent instead keeps every re-dispatch clean.
            Defaults to False since the standalone gemini_agent/
            third_party_agent instances are never part of a sub-agent
            hierarchy at all.

    Returns:
        A configured ADK Agent with the weather tools and logging/
        validation callbacks attached.
    """
    return Agent(
        name=name,
        model=model,
        description="Provides weather information for specific US cities.",
        instruction=AGENT_INSTRUCTION,
        tools=[get_location_lat_long, get_current_weather],
        disallow_transfer_to_peers=disallow_transfer_to_peers,
        before_model_callback=[validate_user_input, log_user_prompt],
        after_model_callback=log_model_response,
    )


gemini_agent = build_weather_agent("weather_agent_gemini", MODEL_GEMINI)
third_party_agent = build_weather_agent("weather_agent_third_party", MODEL_THIRD_PARTY)

print(f"Created '{gemini_agent.name}' and '{third_party_agent.name}' with callbacks attached.")

## Step 6: Wrap Both Agents, Create Sessions

Unchanged from Challenge 2/3/4.

In [ ]:
from vertexai.preview import reasoning_engines

gemini_app = reasoning_engines.AdkApp(agent=gemini_agent)
third_party_app = reasoning_engines.AdkApp(agent=third_party_agent)

user_id = "test-user-id"
gemini_session = gemini_app.create_session(user_id=user_id)
third_party_session = third_party_app.create_session(user_id=user_id)

print(f"Gemini session: {gemini_session['id']}")
print(f"Third-party session: {third_party_session['id']}")

## Step 7: Query Helper

Unchanged from Challenge 1/2/3/4.

In [ ]:
def call_weather_agent(app, session_id: str, prompt: str) -> str:
    """Send a prompt to a weather agent app and return its final text reply.

    Args:
        app: An AdkApp instance (either the Gemini or third-party app).
        session_id: The session ID to use for this query.
        prompt: The user's natural-language query.

    Returns:
        The agent's final text response, or a fallback message if none was
        produced.
    """
    response = "sorry, I have no response"
    for event in app.stream_query(user_id=user_id, session_id=session_id, message=prompt):
        content = event.get("content", {})
        parts = content.get("parts", [])
        if parts and "text" in parts[0]:
            response = parts[0]["text"]
    return response

## Step 8: Test — Multiple US Cities, Both Models

Unchanged from Challenge 1/2/3/4 — still passing before we deploy.

In [ ]:
test_cities = [
    "What is the weather in Miami, FL?",
    "What is the weather in Aspen, Colorado?",
    "What is the weather in Knoxville, TN?",
]

print("=== Gemini-backed agent ===\n")
for prompt in test_cities:
    print(f"user: {prompt}")
    print(f"agent: {call_weather_agent(gemini_app, gemini_session['id'], prompt)}\n")

print("=== Third-party-backed agent (via SAIC gateway) ===\n")
for prompt in test_cities:
    print(f"user: {prompt}")
    print(f"agent: {call_weather_agent(third_party_app, third_party_session['id'], prompt)}\n")

## Step 9: Standalone Search Agent + Root Agent

Unchanged from Challenge 3/4. `search_agent` is wrapped in `AgentTool` (added to `root_agent`'s `tools=[...]`) rather than `sub_agents=[...]` — a pure-`google_search` agent can never work as a plain `sub_agents` member (see Challenge 3 for the full account). `weather_agent_for_root` stays on plain `sub_agents`, with `disallow_transfer_to_peers=True`.

In [ ]:
search_agent = Agent(
    name="search_agent",
    model=MODEL_GEMINI,  # google_search requires Gemini; can't use MODEL_THIRD_PARTY here.
    description="Answers general knowledge and current-events questions using Google Search.",
    instruction=(
        "Use your search tool to look up current, accurate information and "
        "answer the user's question clearly and concisely."
    ),
    tools=[google_search],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

weather_agent_for_root = build_weather_agent(
    "weather_agent_for_root", MODEL_GEMINI, disallow_transfer_to_peers=True
)

root_agent = Agent(
    name="root_agent",
    model=MODEL_GEMINI,
    description="Routes user questions to the appropriate specialist.",
    instruction=(
        "You coordinate between two specialist capabilities and do not "
        "answer questions directly yourself.\n"
        "- For weather questions about US cities, transfer to "
        "'weather_agent_for_root'.\n"
        "- For general knowledge or current-events questions, use your "
        "'search_agent' tool."
    ),
    tools=[AgentTool(agent=search_agent, skip_summarization=False)],
    sub_agents=[weather_agent_for_root],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

root_app = reasoning_engines.AdkApp(agent=root_agent)
root_session = root_app.create_session(user_id=user_id)

print(f"Created '{root_agent.name}' with sub_agents: {[a.name for a in root_agent.sub_agents]}")
print(f"Root agent session: {root_session['id']}")

## Step 10: Answer-Team Agents — Search, Critique, Refine

Unchanged from Challenge 4.

In [ ]:
team_search_agent = Agent(
    name="team_search_agent",
    model=MODEL_GEMINI,
    description="Finds data to answer the user's question.",
    instruction=(
        "The user has asked a question. Use your search tool to find "
        "accurate, current information that answers it, then provide a "
        "clear, direct initial answer."
    ),
    tools=[google_search],
    output_key="initial_answer",
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

critique_agent = Agent(
    name="critique_agent",
    model=MODEL_GEMINI,
    description="Reviews the initial answer and suggests improvements.",
    instruction="""
    Review the INITIAL_ANSWER below. Suggest specific, concrete
    improvements: is it accurate, complete, well-organized, and clearly
    written? List the improvements you'd like to see made.

    INITIAL_ANSWER:
    { initial_answer? }
    """,
    output_key="critique",
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

refine_agent = Agent(
    name="refine_agent",
    model=MODEL_GEMINI,
    description="Rewrites the answer incorporating the critique's suggestions.",
    instruction="""
    Rewrite the INITIAL_ANSWER below, incorporating the improvements
    suggested in CRITIQUE. Produce a single, polished, final answer for
    the user — do not mention the critique process itself in your reply.

    INITIAL_ANSWER:
    { initial_answer? }

    CRITIQUE:
    { critique? }
    """,
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

print(f"Created '{team_search_agent.name}', '{critique_agent.name}', '{refine_agent.name}'.")

## Step 11: Answer Team — SequentialAgent

Unchanged from Challenge 4.

In [ ]:
answer_team = SequentialAgent(
    name="answer_team",
    description="Researches, critiques, and refines an answer to the user's question.",
    sub_agents=[team_search_agent, critique_agent, refine_agent],
)

print(f"Created '{answer_team.name}' with sub_agents: {[a.name for a in answer_team.sub_agents]}")

## Step 12: Greeter Agent

Unchanged from Challenge 4. This is the agent that gets deployed in Step 16 below.

In [ ]:
greeter_agent = Agent(
    name="greeter_agent",
    model=MODEL_GEMINI,
    description="Greets the user and routes their question to the answer team.",
    instruction=(
        "Greet the user and ask what question they'd like answered. Once "
        "they provide a question, transfer to 'answer_team' to research, "
        "critique, and refine a high-quality answer."
    ),
    sub_agents=[answer_team],
    before_model_callback=log_user_prompt,
    after_model_callback=log_model_response,
)

greeter_app = reasoning_engines.AdkApp(agent=greeter_agent)
greeter_session = greeter_app.create_session(user_id=user_id)

print(f"Created '{greeter_agent.name}'.")
print(f"Greeter session: {greeter_session['id']}")

## Step 13: Event-Printing Query Helper

Unchanged from Challenge 4.

In [ ]:
def call_greeter_agent(prompt: str) -> str:
    """Send a prompt to the LOCAL greeter agent, printing every event's
    author so the search -> critique -> refine pipeline is directly
    visible, then return the final text.

    Args:
        prompt: The user's message.

    Returns:
        The final text response from whichever agent produced the last
        text output in the turn.
    """
    response = "sorry, I have no response"
    print(f"--- events for prompt: '{prompt}' ---")
    for event in greeter_app.stream_query(user_id=user_id, session_id=greeter_session["id"], message=prompt):
        author = event.get("author", "unknown")
        content = event.get("content", {})
        parts = content.get("parts", [])
        for part in parts:
            if "text" in part:
                print(f"  [event] author={author}: text='{part['text'][:80]}'")
                response = part["text"]
            else:
                print(f"  [event] author={author}: {part}")
    print()
    return response

## Step 14: Test — Greeter and Answer Team (Local)

Unchanged from Challenge 4 — confirms the agent still works locally before we deploy it.

In [ ]:
greeter_test_prompts = [
    "hello",
    "What caused the fall of the Roman Empire?",
]

for prompt in greeter_test_prompts:
    print(f"user: {prompt}")
    print(f"agent: {call_greeter_agent(prompt)}\n")

## Step 15: Deploy `greeter_agent` to Agent Platform — CHALLENGE 5 ENHANCEMENT

Uses `vertexai.agent_engines.create()` — the Python SDK equivalent of the `adk deploy agent_engine` CLI command used in the GENAI107 lab. Needs an explicit `vertexai.init(...)` call with a staging bucket, since (unlike ambient Gemini calls) deployment needs somewhere to upload the packaged agent artifacts to.

`requirements` only needs `google-cloud-aiplatform[agent_engines,adk]` — no `litellm`, since `greeter_agent`'s entire tree is Gemini-only. This takes several minutes; the deploy call blocks until it completes.

In [ ]:
# === CHALLENGE 5 ENHANCEMENT ===

import vertexai
from vertexai import agent_engines

vertexai.init(
    project=PROJECT_ID,
    location="us-central1",
    staging_bucket=f"gs://{PROJECT_ID}-bucket",
)

remote_greeter_agent = agent_engines.create(
    greeter_agent,
    requirements=[
        "google-cloud-aiplatform[agent_engines,adk]==1.156.0",
        "cloudpickle",
    ],
    display_name="Greeter Agent - Search Critique Refine",
)

print(f"Deployed: {remote_greeter_agent.resource_name}")

## Step 16: Test the Deployed Agent — CHALLENGE 5 ENHANCEMENT

Same two-turn conversation as the local test in Step 14, but now against the deployed remote instance — confirms the deployment actually works end to end, not just that the code compiled.

In [ ]:
# === CHALLENGE 5 ENHANCEMENT ===

remote_session = remote_greeter_agent.create_session(user_id=user_id)
print(f"Remote session: {remote_session['id']}\n")

for prompt in greeter_test_prompts:
    print(f"user: {prompt}")
    response = "sorry, I have no response"
    for event in remote_greeter_agent.stream_query(
        user_id=user_id, session_id=remote_session["id"], message=prompt
    ):
        content = event.get("content", {})
        parts = content.get("parts", [])
        if parts and "text" in parts[0]:
            response = parts[0]["text"]
    print(f"agent: {response}\n")

## Step 17: Cleanup (Optional) — CHALLENGE 5 ENHANCEMENT

Deployed Agent Engine resources keep costing money until deleted — same lesson as GENAI107's Task 4. Run this once you're done testing, or leave it if the deployment is being graded and needs to stay up.

In [ ]:
# === CHALLENGE 5 ENHANCEMENT ===

# Uncomment to delete the deployed agent once you're done with it:
# remote_greeter_agent.delete(force=True)
# print("Deleted remote agent.")

## Step 18: Upload to GitHub

Save this notebook and push it to the `agent-dev-skills-workshop-jay-watson` repository for grading.